# AR Cloud Multi-Agent RAG System

This notebook demonstrates a multi-agent RAG system with department routing, ChromaDB, OpenAI, and Langfuse.

## 1. Setup and Imports

In [1]:
from src.config import CHROMA_PERSIST_DIRECTORY
from src.retrieval.chunking import DEPARTMENT_FOLDERS
from src.retrieval.retriever import retrieve_documents
from src.router import route_question
from src.agents.department_agent import answer_department_question
from src.orchestrator import process_question

print('Chroma directory:', CHROMA_PERSIST_DIRECTORY)

Chroma directory: /Users/cristianarielachetoni/code/AI Engeenering/M3/department-rag-router/storage/chroma


## 2. Document Collections and Vector Stores

Each department has an independent ChromaDB collection. Run the indexing command before executing retrieval examples.

In [2]:
for department in DEPARTMENT_FOLDERS:
    print(department, f'{department}_documents')

hr hr_documents
technology technology_documents
finance finance_documents


## 3. Specialized Department Agents

In [3]:
question = 'How do I request annual leave?'
documents = retrieve_documents(question, 'hr', k=3)
print('Retrieved chunks:', len(documents))
print('Document IDs:', [doc.metadata['document_id'] for doc in documents])

Retrieved chunks: 3
Document IDs: ['HR-002', 'HR-002', 'HR-001']


## 4. Router and Orchestrator

In [4]:
routing = route_question(question)
print('Department:', routing.department)
print('Reason:', routing.reason)

Department: hr
Reason: The question is about requesting annual leave, which is an employee leave policy and HR procedure.


In [5]:
result = process_question(question)
print('Answer:', result['answer'])
print('Sources:', result['sources'])

Answer: To request annual leave at AR Cloud:

1. Submit your vacation request through the employee portal.
2. Include:
   - the first day of absence,
   - the last day of absence,
   - the colleague responsible for urgent handover items.
3. Submit the request at least 10 working days before your first vacation day.
4. You should discuss the planned absence with your manager before submitting, especially if it overlaps with a product release or another team member’s approved absence.
Sources: [{'document_id': 'HR-002', 'source': 'data/hr_docs/leave_policy.md'}, {'document_id': 'HR-002', 'source': 'data/hr_docs/leave_policy.md'}, {'document_id': 'HR-001', 'source': 'data/hr_docs/vacation_policy.md'}]


## 5. Tests and Examples

In [6]:
from pathlib import Path
from src.evaluation import load_test_queries

queries = load_test_queries(Path('test_queries.json'))
print('Available evaluation queries:', len(queries))
for query in queries[:3]:
    print(query['id'], query['question'])

Available evaluation queries: 15
HR-01 How do I request annual vacation leave?
HR-02 How can I request a recurring remote work arrangement?
HR-03 Who is responsible for mandatory employee training?


## 6. Langfuse Observability

The orchestrator records routing, retrieval, and generation traces in Langfuse.

Run the orchestration cells above and inspect the resulting traces in the configured Langfuse project.